# Домашнее задание №3

## Контуры, границы, преобразование Хафа

### Цель

Освоить выделение границ, контурный анализ и поиск геометрических примитивов преобразованием Хафа; научиться показывать, как параметры детектора и разрешение аккумулятора влияют на полноту и точность поиска. Результат работы — не набор картинок с найденными линиями, а измеренная зависимость precision/recall от параметров при контролируемом уровне помех.

[Методические указания блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

## 1. Что используется в работе

| Библиотека | Роль в работе |
|---|---|
| `opencv-python` (`cv2`) | `Sobel`, `Canny`, `findContours`, `HoughLines`, `HoughLinesP`, `HoughCircles` |
| `numpy` | синтетические сцены, помехи, вычисления |
| `scikit-image` | тестовые изображения (`coins`, `camera`, `checkerboard`) |
| `matplotlib` | визуализация и графики серий |
| `pandas` | журнал экспериментов |

Данные: `skimage.data` и синтетическая сцена с **известными** параметрами примитивов (прямые заданы парами $(\rho, \theta)$, окружности — центрами и радиусами). Известная разметка нужна для того, чтобы считать precision и recall, а не оценивать результат по внешнему виду. Интернет не требуется.

Заполните шапку работы (ФИО, группа, версии, seed) согласно п. 1 [общих МУ](../../../docs/guidelines-students.md).

In [ ]:
# Служебная ячейка: импорты, версии, seed.
import time

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import skimage
from skimage import data as skdata

SEED = 42
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

AUTHOR = {"fio": "", "group": "", "work": "ДЗ3"}   # TODO: заполните

VERSIONS = {
    "opencv": cv2.__version__,
    "numpy": np.__version__,
    "scikit-image": skimage.__version__,
    "pandas": pd.__version__,
    "seed": SEED,
}
VERSIONS

## 2. Краткая теоретическая справка

### 2.1. Градиент и оператор Собеля

Граница — область резкого изменения яркости, то есть большого модуля градиента

$$ \nabla I = \left( \frac{\partial I}{\partial x}, \frac{\partial I}{\partial y} \right), \qquad
|\nabla I| = \sqrt{I_x^2 + I_y^2}, \qquad \theta = \operatorname{atan2}(I_y, I_x) $$

Оператор Собеля аппроксимирует производные свёрткой с ядрами

$$ K_x = \begin{pmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{pmatrix}, \qquad
K_y = \begin{pmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{pmatrix} $$

Ядро совмещает дифференцирование по одной оси со сглаживанием по другой, что снижает чувствительность к шуму. Результат нужно вычислять в знаковом типе (`cv2.CV_64F` или `CV_16S`): при `uint8` отрицательная часть градиента обнуляется, и половина границ теряется.

### 2.2. Детектор Кэнни

Четыре этапа: (1) гауссово сглаживание; (2) вычисление модуля и направления градиента; (3) подавление немаксимумов вдоль направления градиента — даёт границы толщиной в один пиксель; (4) двойная пороговая обработка с гистерезисом: пиксели выше $T_{high}$ — границы, ниже $T_{low}$ — фон, промежуточные — границы только при связности с сильными.

Гистерезис — причина того, что пороги нельзя выбирать независимо. Рекомендуемое отношение $T_{high} / T_{low}$ лежит в диапазоне от 2 до 3. Слишком высокий $T_{high}$ рвёт контуры, слишком низкий $T_{low}$ вытягивает шум.

### 2.3. Характеристики контура

Для замкнутого контура: площадь $A$, периметр $P$, компактность (циркулярность)

$$ C = \frac{4\pi A}{P^2} \in (0, 1] $$

($C = 1$ для круга), заполненность бокса $\mathrm{extent} = A / (w h)$, выпуклость $\mathrm{solidity} = A / A_{\text{hull}}$. Аппроксимация полигоном (алгоритм Дугласа — Пекера) управляется параметром $\varepsilon$, обычно задаваемым как доля периметра: $\varepsilon = k P$. Число вершин аппроксимации — базовый признак классификации примитивов (треугольник, четырёхугольник и т. д.), но он крайне чувствителен к $k$.

### 2.4. Преобразование Хафа для прямых

Прямая представляется в нормальной форме

$$ \rho = x \cos\theta + y \sin\theta, \qquad \theta \in [0, \pi), \; \rho \in \mathbb{R} $$

Параметризация $y = kx + b$ не используется: вертикальные прямые дают $k \to \infty$.

Каждый краевой пиксель $(x, y)$ «голосует» за все пары $(\rho, \theta)$, удовлетворяющие уравнению, — то есть за синусоиду в пространстве параметров. **В аккумуляторе хранится число проголосовавших краевых пикселей для каждой дискретной ячейки $(\rho, \theta)$.** Локальные максимумы аккумулятора соответствуют прямым.

Разрешение аккумулятора задаётся шагами $\Delta\rho$ (в пикселях) и $\Delta\theta$ (в радианах). Крупный шаг: голоса от одной прямой собираются в одну ячейку, максимум выражен, но точность параметров низка и близкие прямые сливаются. Мелкий шаг: точность выше, но голоса одной прямой размазываются по соседним ячейкам, максимум падает ниже порога, и прямая теряется. Порог накопления не переносится между разрешениями и разрешениями изображения: он зависит от числа краевых пикселей.

### 2.5. Преобразование Хафа для окружностей

Окружность задаётся тремя параметрами $(x_c, y_c, r)$, аккумулятор трёхмерен. `cv2.HoughCircles` использует градиентный метод: сначала по направлениям градиента накапливаются центры (двумерный аккумулятор), затем для каждого центра оценивается радиус. Ключевые параметры: `param1` — верхний порог Кэнни внутри метода, `param2` — порог аккумулятора центров (чем меньше, тем больше ложных окружностей), `minDist` — минимальное расстояние между центрами, `minRadius` / `maxRadius` — диапазон радиусов.

### 2.6. Метрики поиска примитивов

Обнаружение сопоставляется с эталоном по допуску: прямая считается найденной, если $|\Delta\rho| < \tau_\rho$ и угловое расхождение $< \tau_\theta$; окружность — если расстояние между центрами и разность радиусов меньше допусков. Далее

$$ \text{precision} = \frac{TP}{TP + FP}, \qquad \text{recall} = \frac{TP}{TP + FN}, \qquad
F_1 = \frac{2\,\text{precision}\cdot\text{recall}}{\text{precision} + \text{recall}} $$

Каждый эталонный примитив сопоставляется не более чем одному обнаружению: без этого дубликаты одной и той же прямой искусственно завышают recall.

## 3. Задачи

Формулировка по [методическим указаниям блока](README.md).

1. Выделите границы (Собель, Кэнни) и сравните чувствительность к параметрам.
2. Найдите и проанализируйте контуры (площадь, периметр, аппроксимация, выпуклая оболочка).
3. Найдите прямые и окружности преобразованием Хафа на изображениях с искусственными помехами.

**Ожидаемый результат:** исследование влияния параметров детектора границ и аккумулятора Хафа на полноту/точность поиска примитивов.

**Что будет проверяться** ([рубрика](../teachers-assessment/README.md)): исследована чувствительность Кэнни к порогам; характеристики контуров посчитаны; Хаф находит примитивы при наличии помех; показана зависимость результата от разрешения аккумулятора. Типичная ошибка: пороги Кэнни и параметры Хафа подобраны под один кадр и выдаются за универсальные.

## 4. Данные

Синтетическая сцена содержит прямые и окружности с известными параметрами и управляемый уровень помех трёх видов: гауссов шум, импульсные точки и отвлекающие короткие отрезки. Именно на ней считаются precision/recall.

Реальные изображения (`coins`, `checkerboard`, `camera`) используются для качественного анализа: на них видно, как ведут себя те же параметры на материале с текстурой, тенями и бликами.

In [ ]:
# Служебная ячейка: синтетическая сцена с известной разметкой. Изменять не требуется.

def normalize_line(rho: float, theta: float) -> tuple:
    '''Привести (rho, theta) к канону: theta in [0, pi), rho может быть любого знака.'''
    theta = float(theta) % (2 * np.pi)
    rho = float(rho)
    if theta >= np.pi:
        theta -= np.pi
        rho = -rho
    return rho, theta


def draw_line_rho_theta(canvas: np.ndarray, rho: float, theta: float,
                        color: int = 255, thickness: int = 2) -> None:
    '''Нарисовать прямую, заданную нормальной формой, на всю ширину кадра.'''
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a * rho, b * rho
    length = 2000
    p1 = (int(round(x0 - length * b)), int(round(y0 + length * a)))
    p2 = (int(round(x0 + length * b)), int(round(y0 - length * a)))
    cv2.line(canvas, p1, p2, color, thickness)


def make_primitive_scene(size: int = 400, n_lines: int = 3, n_circles: int = 3,
                         noise_sigma: float = 0.0, impulse: float = 0.0,
                         n_distractors: int = 0, seed: int = SEED) -> tuple:
    '''Сцена с прямыми и окружностями и управляемым уровнем помех.

    Вход:
        noise_sigma    : сила гауссова шума (единицы шкалы 0..255)
        impulse        : доля импульсных пикселей
        n_distractors  : число коротких отвлекающих отрезков
    Выход:
        (image uint8 [size, size],
         lines   list[(rho, theta)] — эталонные прямые,
         circles list[(x, y, r)]    — эталонные окружности)
    '''
    rng = np.random.default_rng(seed)
    img = np.zeros((size, size), dtype=np.uint8)

    lines = []
    attempts = 0
    while len(lines) < n_lines and attempts < 500:
        attempts += 1
        theta = float(rng.uniform(0, np.pi))
        rho = float(rng.uniform(-0.9 * size, 0.9 * size))
        rho, theta = normalize_line(rho, theta)
        # прямая должна пересекать кадр достаточно длинной хордой
        probe = np.zeros_like(img)
        draw_line_rho_theta(probe, rho, theta, color=255, thickness=2)
        if int((probe > 0).sum()) < int(1.5 * size):
            continue
        # и не дублировать уже принятую прямую
        too_close = any(min(abs(theta - t), np.pi - abs(theta - t)) < np.deg2rad(12)
                        and abs(rho - r) < 0.15 * size for r, t in lines)
        if too_close:
            continue
        img = np.maximum(img, probe)
        lines.append((rho, theta))

    circles = []
    for _ in range(n_circles):
        r = int(rng.integers(25, 55))
        cx = int(rng.integers(r + 10, size - r - 10))
        cy = int(rng.integers(r + 10, size - r - 10))
        cv2.circle(img, (cx, cy), r, 255, 2)
        circles.append((float(cx), float(cy), float(r)))

    for _ in range(n_distractors):
        p1 = tuple(int(v) for v in rng.integers(0, size, size=2))
        offset = rng.integers(-30, 30, size=2)
        p2 = tuple(int(np.clip(p1[i] + offset[i], 0, size - 1)) for i in range(2))
        cv2.line(img, p1, p2, 255, 1)

    out = img.astype(np.float64)
    if noise_sigma > 0:
        out += rng.normal(0.0, noise_sigma, out.shape)
    if impulse > 0:
        mask = rng.random(out.shape)
        out[mask < impulse / 2] = 255
        out[(mask >= impulse / 2) & (mask < impulse)] = 0
    return np.clip(out, 0, 255).astype(np.uint8), lines, circles


SCENES = {
    "clean": make_primitive_scene(seed=SEED),
    "noisy": make_primitive_scene(noise_sigma=25.0, impulse=0.01, n_distractors=15, seed=SEED),
    "hard": make_primitive_scene(noise_sigma=45.0, impulse=0.03, n_distractors=40, seed=SEED),
}
REAL = {
    "coins": skdata.coins().astype(np.uint8),
    "checkerboard": skdata.checkerboard().astype(np.uint8),
    "camera": skdata.camera().astype(np.uint8),
}
print("Эталон сцены clean: прямых", len(SCENES["clean"][1]), ", окружностей", len(SCENES["clean"][2]))

In [ ]:
# Служебная ячейка: визуализация, журнал, сопоставление с эталоном. Изменять не требуется.
RUNS: list = []


def log_run(**fields) -> dict:
    row = {"seed": SEED, **fields}
    if isinstance(row.get("params"), dict):
        row["params"] = ", ".join(f"{k}={v}" for k, v in row["params"].items())
    RUNS.append(row)
    return row


def runs_table(stage: str = None) -> pd.DataFrame:
    df = pd.DataFrame(RUNS)
    if stage is not None and not df.empty:
        df = df[df["stage"] == stage]
    return df.reset_index(drop=True)


def show_row(images, titles=None, figsize_scale: float = 3.4) -> None:
    n = len(images)
    titles = titles or [""] * n
    fig, axes = plt.subplots(1, n, figsize=(figsize_scale * n, figsize_scale))
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap="gray", vmin=0, vmax=255) if img.ndim == 2 else ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def prf(tp: int, fp: int, fn: int) -> dict:
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}


def match_lines(detected, ground_truth, rho_tol: float = 12.0,
                theta_tol_deg: float = 5.0) -> dict:
    '''Сопоставить найденные прямые с эталонными (жадно, один к одному).

    Вход: последовательности пар (rho, theta) в канонической форме.
    Выход: dict с tp, fp, fn, precision, recall, f1.
    '''
    theta_tol = np.deg2rad(theta_tol_deg)
    unused = list(range(len(ground_truth)))
    tp = 0
    for rho_d, theta_d in [normalize_line(r, t) for r, t in detected]:
        best, best_cost = None, None
        for j in unused:
            rho_g, theta_g = normalize_line(*ground_truth[j])
            direct = abs(theta_d - theta_g)
            if direct <= np.pi - direct:
                d_theta, d_rho = direct, abs(rho_d - rho_g)
            else:
                # theta_d и theta_g лежат у разных концов [0, pi): прямые почти
                # параллельны, но знаки rho противоположны
                d_theta, d_rho = np.pi - direct, abs(rho_d + rho_g)
            if d_rho < rho_tol and d_theta < theta_tol:
                cost = d_rho / rho_tol + d_theta / theta_tol
                if best_cost is None or cost < best_cost:
                    best, best_cost = j, cost
        if best is not None:
            unused.remove(best)
            tp += 1
    return prf(tp, len(detected) - tp, len(unused))


def match_circles(detected, ground_truth, center_tol: float = 12.0,
                  radius_tol: float = 10.0) -> dict:
    '''Сопоставить найденные окружности (x, y, r) с эталонными, один к одному.'''
    unused = list(range(len(ground_truth)))
    tp = 0
    for x, y, r in detected:
        best, best_cost = None, None
        for j in unused:
            gx, gy, gr = ground_truth[j]
            d_center = float(np.hypot(x - gx, y - gy))
            d_radius = abs(r - gr)
            if d_center < center_tol and d_radius < radius_tol:
                cost = d_center / center_tol + d_radius / radius_tol
                if best_cost is None or cost < best_cost:
                    best, best_cost = j, cost
        if best is not None:
            unused.remove(best)
            tp += 1
    return prf(tp, len(detected) - tp, len(unused))


show_row([SCENES["clean"][0], SCENES["noisy"][0], SCENES["hard"][0], REAL["coins"]],
         ["сцена clean", "сцена noisy", "сцена hard", "coins"])

## 5. Задание 1. Выделение границ: Собель и Кэнни

Рабочий пример показывает обе ветви на одном изображении. Обратите внимание на тип данных при вызове `cv2.Sobel`: производная знакова, и `cv2.CV_64F` с последующим `np.abs` даёт полную карту, тогда как `uint8` теряет переходы «светлое → тёмное».

Для количественной оценки чувствительности используется синтетическая сцена: эталонная карта границ известна (это сцена без помех), поэтому можно считать F1 с допуском на смещение в 1–2 пикселя. Функция `edge_f1` ниже реализует такое сопоставление через дилатацию.

In [ ]:
# Служебная ячейка + рабочий пример (рельсы): Собель, Кэнни, метрика границ.

def edge_f1(pred_edges: np.ndarray, gt_edges: np.ndarray, tolerance_px: int = 2) -> dict:
    '''F1 для карт границ с допуском на пространственное смещение.

    Вход: две бинарные карты одинакового размера (ненулевое = граница).
    Совпадением считается краевой пиксель, попавший в tolerance_px от эталона.
    '''
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                       (2 * tolerance_px + 1, 2 * tolerance_px + 1))
    pred = (pred_edges > 0).astype(np.uint8)
    gt = (gt_edges > 0).astype(np.uint8)
    gt_dilated = cv2.dilate(gt, kernel)
    pred_dilated = cv2.dilate(pred, kernel)
    tp_p = int(np.logical_and(pred, gt_dilated).sum())
    tp_g = int(np.logical_and(gt, pred_dilated).sum())
    precision = tp_p / max(int(pred.sum()), 1)
    recall = tp_g / max(int(gt.sum()), 1)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
            "edge_pixels": int(pred.sum())}


scene, gt_lines, gt_circles = SCENES["noisy"]
clean_scene = SCENES["clean"][0]
gt_edges = cv2.Canny(clean_scene, 100, 200)      # эталонная карта границ (сцена без помех)

gx = cv2.Sobel(scene, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(scene, cv2.CV_64F, 0, 1, ksize=3)
magnitude = np.clip(np.hypot(gx, gy), 0, 255).astype(np.uint8)
canny = cv2.Canny(scene, 100, 200)

log_run(image="scene_noisy", stage="edges", method="canny",
        params={"t_low": 100, "t_high": 200}, **edge_f1(canny, gt_edges))

show_row([scene, np.abs(gx).clip(0, 255).astype(np.uint8), magnitude, canny],
         ["сцена noisy", "|Собель по x|", "модуль градиента", "Кэнни 100/200"])
runs_table("edges")

In [ ]:
# TODO (задание 1.1): единый интерфейс выделения границ.

def edge_map(image: np.ndarray, method: str, **params) -> np.ndarray:
    '''Построить бинарную карту границ.

    Вход:
        image  : np.ndarray uint8 [H, W]
        method : "sobel" | "canny"
        params : для sobel — ksize, threshold (порог по модулю градиента);
                 для canny — t_low, t_high, aperture_size, blur_ksize (предсглаживание)
    Выход:
        np.ndarray uint8 [H, W], значения 0 или 255.
    Замечания:
        - производные вычисляйте в знаковом типе (cv2.CV_64F);
        - предварительное сглаживание — отдельный фактор, не «зашивайте» его молча
          внутрь метода: его влияние нужно измерить отдельно.
    '''
    raise NotImplementedError

In [ ]:
# TODO (задание 1.2): исследование чувствительности к параметрам.
#
# Обязательная серия для Кэнни (метрика — edge_f1 относительно gt_edges):
#   t_low  in [30, 50, 100, 150]
#   ratio  = t_high / t_low in [2.0, 3.0]
#   blur_ksize in [0, 3, 5]
#   уровень помех: сцены "clean", "noisy", "hard"
# Для Собеля: ksize in [3, 5, 7] и не менее трёх порогов по модулю градиента.
#
# В одном сравнении меняйте один фактор. Порог, найденный на сцене "clean",
# обязательно проверьте на "noisy" и "hard" — это и есть проверка того, что
# параметры не подогнаны под один кадр (типичная ошибка по рубрике).
#
# Каркас:
# for scene_name, (img, _, _) in SCENES.items():
#     for t_low in (...):
#         for ratio in (...):
#             edges = edge_map(img, "canny", t_low=t_low, t_high=int(t_low * ratio))
#             log_run(image=scene_name, stage="edges", method="canny",
#                     params={"t_low": t_low, "ratio": ratio}, **edge_f1(edges, gt_edges))
#
# Дополнительно: примените лучшие по метрике параметры к REAL["coins"] и
# REAL["camera"] и покажите визуально, переносятся ли они на реальный материал.

# TODO: код серии

runs_table("edges")

## 6. Задание 2. Контуры: поиск и анализ

`cv2.findContours` работает с бинарным изображением, поэтому качество контуров полностью определяется качеством предшествующей бинаризации. Режим `cv2.RETR_EXTERNAL` возвращает только внешние контуры, `cv2.RETR_TREE` — с иерархией вложенности; метод `cv2.CHAIN_APPROX_SIMPLE` сжимает прямые участки.

Рабочий пример: бинаризация `coins` методом Оцу, морфологическое закрытие, поиск внешних контуров и вычисление характеристик первого из них.

In [ ]:
# Рабочий пример (рельсы): контуры монет и характеристики одного контура.
coins = REAL["coins"]
blurred = cv2.GaussianBlur(coins, (5, 5), 0)
_, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,
                          cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))

contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print("Найдено контуров:", len(contours))

c = max(contours, key=cv2.contourArea)
area = cv2.contourArea(c)
perimeter = cv2.arcLength(c, True)
hull = cv2.convexHull(c)
approx = cv2.approxPolyDP(c, 0.02 * perimeter, True)
print(f"Крупнейший контур: площадь={area:.1f}, периметр={perimeter:.1f}, "
      f"циркулярность={4 * np.pi * area / perimeter ** 2:.3f}, "
      f"вершин аппроксимации={len(approx)}, solidity={area / cv2.contourArea(hull):.3f}")

canvas = cv2.cvtColor(coins, cv2.COLOR_GRAY2RGB)
cv2.drawContours(canvas, contours, -1, (255, 0, 0), 2)
cv2.drawContours(canvas, [hull], -1, (0, 160, 255), 2)
show_row([coins, binary, canvas], ["coins", "бинаризация Оцу + закрытие", "контуры и оболочка"])

In [ ]:
# TODO (задание 2.1): характеристики контура.

def contour_features(contour: np.ndarray, approx_eps_ratio: float = 0.02) -> dict:
    '''Вычислить характеристики одного контура.

    Вход:
        contour : np.ndarray [N, 1, 2] в формате OpenCV
        approx_eps_ratio : доля периметра для approxPolyDP
    Выход: dict с ключами
        area, perimeter, circularity, extent, aspect_ratio, solidity,
        n_vertices (после аппроксимации), centroid (cx, cy).
    Граничные случаи, которые нужно обработать явно:
        - нулевой периметр или нулевая площадь (вырожденный контур);
        - контур из менее чем 5 точек;
        - нулевая площадь выпуклой оболочки.
    '''
    raise NotImplementedError

In [ ]:
# TODO (задание 2.2): контурный анализ набора изображений.
#
# Требования:
# 1. Построить таблицу характеристик всех контуров изображения (pandas.DataFrame)
#    и отфильтровать шумовые контуры по площади; порог фильтрации обосновать
#    гистограммой площадей, а не подобрать «на глаз».
# 2. Показать влияние параметра approxPolyDP: eps_ratio in [0.005, 0.01, 0.02, 0.05]
#    — число вершин и визуальный результат на одном и том же контуре.
# 3. Показать различие контура и его выпуклой оболочки на невыпуклом объекте
#    (используйте sk.data или синтетическую фигуру с вырезом) и вычислить solidity.
# 4. Классифицировать примитивы синтетической сцены по признакам
#    (circularity, n_vertices) и оценить, где классификация ошибается.
# 5. Повторить анализ на "noisy"/"hard" и показать, как помехи меняют
#    число и характеристики контуров.
#
# Записи в журнал: log_run(stage="contours", image=..., method=..., params=..., ...)

# TODO: код заданий 2.2

## 7. Задание 3. Преобразование Хафа: прямые

`cv2.HoughLines` возвращает массив формы `[N, 1, 2]` с парами $(\rho, \theta)$ — это стандартное преобразование по всему аккумулятору. `cv2.HoughLinesP` — вероятностный вариант, возвращающий отрезки `[x1, y1, x2, y2]`; он экономнее, но его выход нельзя напрямую сопоставить с эталоном в $(\rho, \theta)$ без пересчёта.

Порог `threshold` — это минимальное число голосов, то есть число краевых пикселей на прямой. Он зависит от размера изображения, толщины линий и параметров детектора границ, поэтому переносить его между сценами без пересчёта нельзя.

Рабочий пример: один запуск на зашумлённой сцене с сопоставлением результата с эталоном.

In [ ]:
# Рабочий пример (рельсы): HoughLines и сопоставление с эталоном.
scene, gt_lines, gt_circles = SCENES["noisy"]
edges = cv2.Canny(cv2.GaussianBlur(scene, (5, 5), 0), 80, 200)

raw = cv2.HoughLines(edges, rho=1, theta=np.pi / 180, threshold=140)
detected = [] if raw is None else [normalize_line(r, t) for r, t in raw[:, 0, :]]
print("Найдено прямых:", len(detected), "| эталонных:", len(gt_lines))

metrics = match_lines(detected, gt_lines)
log_run(image="scene_noisy", stage="hough_lines", method="HoughLines",
        params={"rho": 1, "theta_deg": 1, "threshold": 140}, n_detected=len(detected), **metrics)

canvas = cv2.cvtColor(scene, cv2.COLOR_GRAY2RGB)
layer = np.zeros(scene.shape, dtype=np.uint8)      # рисуем в отдельный слой:
for rho, theta in detected:                        # cv2 не пишет в срез-представление
    draw_line_rho_theta(layer, rho, theta, color=255, thickness=1)
canvas[layer > 0] = (255, 0, 0)
show_row([scene, edges, canvas], ["сцена noisy", "границы (Кэнни)", "найденные прямые (красный)"])
metrics

In [ ]:
# TODO (задание 3.1): обёртка поиска прямых.

def detect_lines(image: np.ndarray, *, canny_params: dict, rho: float = 1.0,
                 theta: float = np.pi / 180, threshold: int = 150) -> list:
    '''Найти прямые преобразованием Хафа.

    Вход:
        image        : uint8 [H, W]
        canny_params : dict параметров детектора границ (t_low, t_high, blur_ksize)
        rho, theta   : разрешение аккумулятора (пиксели и радианы)
        threshold    : порог накопления
    Выход:
        list[(rho, theta)] в канонической форме (используйте normalize_line).
    Обработайте случай, когда cv2.HoughLines вернул None.
    '''
    raise NotImplementedError

In [ ]:
# TODO (задание 3.2): зависимость от разрешения аккумулятора и уровня помех.
#
# Обязательная серия (метрика — match_lines: precision, recall, f1):
#   rho_step   in [1, 2, 4, 8]              (пиксели)
#   theta_step in [0.5, 1, 2, 4] градусов   (переводите в радианы)
#   threshold  in [80, 120, 160, 200]
#   сцены      in {"clean", "noisy", "hard"}
# Правило серии: варьируйте один параметр аккумулятора, остальные фиксируйте.
#
# Обязательные к демонстрации эффекты (это материал для защиты):
# 1. Слишком мелкий шаг по theta при фиксированном threshold: голоса одной прямой
#    размазываются, recall падает.
# 2. Слишком крупный шаг: близкие прямые сливаются, точность параметров падает
#    (посмотрите на среднюю ошибку |d_rho| у истинно положительных).
# 3. Порог, подобранный на "clean", применённый к "hard": покажите деградацию.
#
# Дополнительно: сравните HoughLines и HoughLinesP по числу ложных срабатываний
# и по времени; для HoughLinesP пересчитайте отрезки в (rho, theta) для сопоставления.

# TODO: код серии

runs_table("hough_lines")

## 8. Задание 3. Преобразование Хафа: окружности

`cv2.HoughCircles` требует полутонового входа (не карты границ) и внутренне вызывает Кэнни с порогом `param1`. Метод чувствителен к `param2` и `minDist`: заниженный `param2` даёт много ложных окружностей, завышенный — пропуски; `minDist` меньше диаметра объектов порождает дубликаты вокруг одного объекта.

Предварительное сглаживание для окружностей обычно обязательно: без него градиентные направления зашумлены, и аккумулятор центров размывается.

In [ ]:
# Рабочий пример (рельсы): HoughCircles на синтетической сцене.
scene, gt_lines, gt_circles = SCENES["noisy"]
blurred = cv2.GaussianBlur(scene, (5, 5), 1.5)

raw = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1.0, minDist=30,
                       param1=150, param2=45, minRadius=20, maxRadius=60)
detected = [] if raw is None else [tuple(float(v) for v in c) for c in raw[0]]
metrics = match_circles(detected, gt_circles)
log_run(image="scene_noisy", stage="hough_circles", method="HoughCircles",
        params={"dp": 1.0, "minDist": 30, "param1": 150, "param2": 45},
        n_detected=len(detected), **metrics)

canvas = cv2.cvtColor(scene, cv2.COLOR_GRAY2RGB)
for x, y, r in detected:
    cv2.circle(canvas, (int(x), int(y)), int(r), (255, 0, 0), 2)
for x, y, r in gt_circles:
    cv2.circle(canvas, (int(x), int(y)), int(r), (0, 200, 255), 1)
show_row([scene, canvas], ["сцена noisy", "найдено (красный) / эталон (оранжевый)"])
metrics

In [ ]:
# TODO (задание 3.3): серия по параметрам поиска окружностей.
#
# Обязательная серия (метрика — match_circles):
#   dp      in [1.0, 1.5, 2.0]     (обратное разрешение аккумулятора центров)
#   param2  in [25, 35, 45, 60]
#   minDist in [15, 30, 60]
#   сцены   in {"clean", "noisy", "hard"}
#
# Показать:
# 1. Компромисс precision/recall по param2 (график precision и recall от param2).
# 2. Эффект dp: dp = 2 означает аккумулятор вдвое меньшего разрешения — как это
#    сказывается на точности центров и на числе пропусков.
# 3. Влияние minDist на дубликаты.
# 4. Перенос лучших параметров на REAL["coins"]: реальные монеты имеют
#    неравномерную заливку и тени. Оцените результат качественно и объясните
#    расхождение с синтетикой.

# TODO: код серии

runs_table("hough_circles")

## Отчёт

**Таблица 1.** Кэнни: пороги × уровень помех → precision/recall/F1 карты границ, число краевых пикселей.

**Таблица 2.** Контуры: изображение → число контуров после фильтрации, распределение характеристик, результат классификации примитивов.

**Таблица 3.** Хаф (прямые): разрешение аккумулятора и порог × сцена → precision/recall/F1, число обнаружений, время.

**Таблица 4.** Хаф (окружности): `param2`, `dp`, `minDist` × сцена → precision/recall/F1.

Обязательные графики: (а) F1 Кэнни от порога для трёх уровней помех; (б) precision и recall Хафа от порога накопления; (в) F1 от шага аккумулятора.

Разделяйте наблюдение, интерпретацию и вывод (п. 2 [общих МУ](../../../docs/guidelines-students.md)). Формулировка «Кэнни с порогами 100/200 работает хорошо» выводом не является: укажите сцену, метрику и диапазон, в котором утверждение проверено.

In [ ]:
# Сводные таблицы из журнала.
for stage in ("edges", "contours", "hough_lines", "hough_circles"):
    df = runs_table(stage)
    if df.empty:
        print(f"{stage}: журнал пуст.")
        continue
    print(f"\n=== {stage} ===")
    cols = [c for c in ("precision", "recall", "f1", "n_detected", "edge_pixels") if c in df.columns]
    display(df.groupby(["image", "method", "params"])[cols].mean().round(3))

# TODO: постройте требуемые графики. Для каждого графика подпишите оси и укажите,
# какие параметры зафиксированы.

### Выводы

**Наблюдения**

1.
2.
3.

**Интерпретация**

1.
2.

**Выводы и границы применимости**

1.
2.

**Анализ отказов.** Обязательно разберите: (а) сцену, на которой параметры, оптимальные для `clean`, дают низкий recall; (б) случай слияния близких прямых при крупном шаге аккумулятора; (в) ложные окружности при заниженном `param2`. Для каждого случая — иллюстрация и объяснение через механизм метода.

**Использование сторонних материалов и LLM.** Укажите источники (п. 5 общих МУ).

## Контрольные вопросы

Из [списка вопросов блока](README.md#контрольные-вопросы-блока), относящиеся к этой работе:

3. Что такое частотная фильтрация и как связаны свёртка и произведение спектров? (в контексте предварительного сглаживания перед выделением границ)
7. Как устроено преобразование Хафа для прямых? Что хранится в аккумуляторе?

Дополнительно к защите: как изменится аккумулятор Хафа при увеличении шага по углу? Почему параметризация $y = kx + b$ непригодна для преобразования Хафа?

## Чек-лист перед сдачей

Полный список — в [общих МУ, п. 6](../../../docs/guidelines-students.md#6-чек-лист-перед-сдачей). Специфика ДЗ3:

- [ ] Ноутбук исполняется сверху вниз без ошибок после `Restart & Run All`.
- [ ] Собель и Кэнни применены, разница между ними объяснена.
- [ ] Чувствительность Кэнни к порогам исследована серией, а не одним запуском.
- [ ] Параметры, подобранные на чистой сцене, проверены на зашумлённой.
- [ ] Посчитаны площадь, периметр, аппроксимация и выпуклая оболочка контуров.
- [ ] Показано влияние `eps` в `approxPolyDP`.
- [ ] Хаф применён к прямым и окружностям на изображениях с помехами.
- [ ] Показана зависимость результата от разрешения аккумулятора ($\Delta\rho$, $\Delta\theta$, `dp`).
- [ ] Precision/recall посчитаны относительно известной разметки, а не оценены визуально.
- [ ] Наблюдения отделены от интерпретаций, границы выводов указаны.